In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import math
from matplotlib import font_manager

In [2]:
lfi_methods = ["lmdi_plus", "lmdi_baseline", "lime", "shap"]

In [3]:
pves = [0.1]
rhos = [0.5, 0.6, 0.7, 0.8, 0.9, 0.99]
mean_results = {}
sd_results = {}
for method in lfi_methods:
    group_results = {}
    sd_group_results = {}
    sig_mat = np.zeros((len(pves), len(rhos)))
    c_nsig_mat = np.zeros((len(pves), len(rhos)))
    nsig_mat = np.zeros((len(pves), len(rhos)))
    sig_sd = np.zeros((len(pves), len(rhos)))
    c_nsig_sd = np.zeros((len(pves), len(rhos)))
    nsig_sd = np.zeros((len(pves), len(rhos)))
    for pve_idx in range(len(pves)):
        for rho_idx in range(len(rhos)):
            rankings = np.zeros((250, 100, 50)) # 250 samples, 100 features, 50 seeds
            for seed in range(1, 51):
                rankings[:, :, seed-1] = pd.read_csv(f"old-results/pve{pves[pve_idx]}/rho{rhos[rho_idx]}/seed{seed}/gb/rankings/{method}.csv").to_numpy()
            sds = np.std(np.mean(rankings, axis = 0), axis=1)/math.sqrt(rankings.shape[0])
            sig_mat[pve_idx, rho_idx] = np.mean(rankings[:, :6])
            sig_sd[pve_idx, rho_idx] = np.mean(sds[:6])
            # average features 7-50 in rankings
            c_nsig_mat[pve_idx, rho_idx] = np.mean(rankings[:, 6:50])
            c_nsig_sd[pve_idx, rho_idx] = np.mean(sds[6:50])
            # average features 51-100 in rankings
            nsig_mat[pve_idx, rho_idx] = np.mean(rankings[:, 50:])
            nsig_sd[pve_idx, rho_idx] = np.mean(sds[50:])
    group_results['sig'] = sig_mat
    group_results['c_nsig'] = c_nsig_mat
    group_results['nsig'] = nsig_mat
    sd_group_results['sig'] = sig_sd
    sd_group_results['c_nsig'] = c_nsig_sd
    sd_group_results['nsig'] = nsig_sd
    sd_results[method] = sd_group_results
    mean_results[method] = group_results

In [4]:
mean_results

{'lmdi_plus': {'sig': array([[46.6052    , 44.82072   , 43.37218667, 42.30481333, 41.32374667,
          37.66      ]]),
  'c_nsig': array([[49.87297455, 50.60695455, 51.52880364, 52.42976   , 54.70008182,
          58.95752727]]),
  'nsig': array([[49.5191584, 49.0873936, 48.4499904, 47.7852336, 45.9050784,
          42.598176 ]])},
 'lmdi_baseline': {'sig': array([[45.82526667, 48.04124   , 50.64438667, 53.88732   , 56.7782    ,
          63.62150667]]),
  'c_nsig': array([[50.26209091, 50.64519636, 51.42012182, 53.21369455, 55.41022182,
          59.99556545]]),
  'nsig': array([[49.270328 , 48.6672784, 47.6729664, 45.7054704, 43.4256208,
          38.5693216]])},
 'lime': {'sig': array([[43.19654667, 44.48704   , 46.96612   , 49.65109333, 52.97302667,
          60.25537333]]),
  'c_nsig': array([[49.59749273, 49.71676727, 49.85194727, 50.59624909, 51.82324545,
          53.75246909]]),
  'nsig': array([[50.1706208, 49.9108   , 49.494352 , 48.5171696, 47.0387808,
          44.467182

In [5]:
# turn mean results into a dataframe
mean_df = pd.DataFrame(columns=['method', 'pve', 'rho', 'sig', 'c_nsig', 'nsig'])
row_list = []
for method in lfi_methods:
    for pve_idx in range(len(pves)):
        for rho_idx in range(len(rhos)):
            row_list.append({
                'method': method,
                'pve': pves[pve_idx],
                'rho': rhos[rho_idx],
                'sig': mean_results[method]['sig'][pve_idx, rho_idx],
                'c_nsig': mean_results[method]['c_nsig'][pve_idx, rho_idx],
                'nsig': mean_results[method]['nsig'][pve_idx, rho_idx]
            })
row_list

[{'method': 'lmdi_plus',
  'pve': 0.1,
  'rho': 0.5,
  'sig': 46.6052,
  'c_nsig': 49.87297454545455,
  'nsig': 49.5191584},
 {'method': 'lmdi_plus',
  'pve': 0.1,
  'rho': 0.6,
  'sig': 44.82072,
  'c_nsig': 50.60695454545454,
  'nsig': 49.0873936},
 {'method': 'lmdi_plus',
  'pve': 0.1,
  'rho': 0.7,
  'sig': 43.372186666666664,
  'c_nsig': 51.528803636363634,
  'nsig': 48.4499904},
 {'method': 'lmdi_plus',
  'pve': 0.1,
  'rho': 0.8,
  'sig': 42.304813333333335,
  'c_nsig': 52.42976,
  'nsig': 47.7852336},
 {'method': 'lmdi_plus',
  'pve': 0.1,
  'rho': 0.9,
  'sig': 41.323746666666665,
  'c_nsig': 54.700081818181815,
  'nsig': 45.9050784},
 {'method': 'lmdi_plus',
  'pve': 0.1,
  'rho': 0.99,
  'sig': 37.66,
  'c_nsig': 58.957527272727276,
  'nsig': 42.598176},
 {'method': 'lmdi_baseline',
  'pve': 0.1,
  'rho': 0.5,
  'sig': 45.825266666666664,
  'c_nsig': 50.26209090909091,
  'nsig': 49.270328},
 {'method': 'lmdi_baseline',
  'pve': 0.1,
  'rho': 0.6,
  'sig': 48.04124,
  'c_nsig

In [6]:
# turn row_list into a dataframe
mean_df = pd.DataFrame(row_list)
# drop pve column
mean_df.drop(columns=['pve'], inplace=True)

In [7]:
mean_df[mean_df['rho'] == 0.99]

,method,rho,sig,c_nsig,nsig
5,lmdi_plus,0.99,37.660000,58.957527,42.598176
11,lmdi_baseline,0.99,63.621507,59.995565,38.569322
17,lime,0.99,60.255373,53.752469,44.467182
23,shap,0.99,62.178093,58.044684,40.459307


In [8]:
# rename method column values to be "LMDI+" "Baseline LMDI" "LIME" "TreeSHAP"
mean_df['method'] = mean_df['method'].replace({
    'lmdi_plus': 'LMDI+',
    'lmdi_baseline': 'Baseline LMDI',
    'lime': 'LIME',
    'shap': 'TreeSHAP'
})
# rename columns to be "Method", "Correlation", "Avg. Signal Feature Ranking", "Avg. Correlated Non-Signal Feature Ranking", "Avg. Non-Signal Feature Ranking"
mean_df.rename(columns={
    'method': 'Method',
    'rho': 'Correlation',
    'sig': 'Avg. Signal Feature Ranking',
    'c_nsig': 'Avg. Correlated Non-Signal Feature Ranking',
    'nsig': 'Avg. Non-Signal Feature Ranking'
}, inplace=True)

In [11]:
# round all values to 2 decimal places
mean_df = mean_df[mean_df['Correlation'] == 0.99].round(2)
# display markdown table
print(mean_df.to_markdown(index=False))

| Method        |   Correlation |   Avg. Signal Feature Ranking |   Avg. Correlated Non-Signal Feature Ranking |   Avg. Non-Signal Feature Ranking |
|:--------------|--------------:|------------------------------:|---------------------------------------------:|----------------------------------:|
| LMDI+         |          0.99 |                         37.66 |                                        58.96 |                             42.6  |
| Baseline LMDI |          0.99 |                         63.62 |                                        60    |                             38.57 |
| LIME          |          0.99 |                         60.26 |                                        53.75 |                             44.47 |
| TreeSHAP      |          0.99 |                         62.18 |                                        58.04 |                             40.46 |


In [ ]:
titles = {'shap': 'TreeSHAP', 'lime': 'LIME', 'lmdi_baseline': 'LMDI',
          'lmdi_plus': 'LMDI+'}
gb_plots = []
for pve_idx in range(len([1])):
    print("--------------------------")
    fig, axs = plt.subplots(1, len(lfi_methods), sharey=True, figsize=(20, 5))
    if pves[pve_idx] == 0.1:
        plt.ylim(30, 70)
    else:
        plt.ylim(10, 70)
    for method_idx in range(len(lfi_methods)):
        # plot results, where each group is a separate line on the plot
        plt.rcParams['axes.labelsize'] = 30
        plt.rcParams['xtick.labelsize'] = 10
        plt.rcParams['ytick.labelsize'] = 12
        plt.rcParams['axes.spines.right'] = False
        plt.rcParams['axes.spines.top'] = False
        # plt.rcParams['axes.edgecolor'] = 'lightgrey'
        plt.rcParams['axes.edgecolor'] = 'black'
        plt.rcParams['axes.linewidth'] = 2.0
        axs[method_idx].plot(mean_results[lfi_methods[method_idx]]['sig'][pve_idx, :], marker = "o",
                 label="Signal", color='forestgreen')
        axs[method_idx].fill_between(range(len(rhos)),
                         mean_results[lfi_methods[method_idx]]['sig'][pve_idx, :] - \
                             sd_results[lfi_methods[method_idx]]['sig'][pve_idx, :],
                             mean_results[lfi_methods[method_idx]]['sig'][pve_idx, :] + \
                                 sd_results[lfi_methods[method_idx]]['sig'][pve_idx, :],
                                 color='forestgreen', alpha=0.3)
        axs[method_idx].plot(mean_results[lfi_methods[method_idx]]['c_nsig'][pve_idx, :], marker = "o",
                 label="Cor. Non-Signal", color='#ADEBB3')
        axs[method_idx].fill_between(range(len(rhos)),
                         mean_results[lfi_methods[method_idx]]['c_nsig'][pve_idx, :] - \
                             sd_results[lfi_methods[method_idx]]['c_nsig'][pve_idx, :],
                             mean_results[lfi_methods[method_idx]]['c_nsig'][pve_idx, :] + \
                                 sd_results[lfi_methods[method_idx]]['c_nsig'][pve_idx, :],
                                 color='#ADEBB3', alpha=0.3)
        axs[method_idx].plot(mean_results[lfi_methods[method_idx]]['nsig'][pve_idx, :], marker = "o",
                 label="Non-Signal", color='red')
        axs[method_idx].fill_between(range(len(rhos)),
                         mean_results[lfi_methods[method_idx]]['nsig'][pve_idx, :] - \
                             sd_results[lfi_methods[method_idx]]['nsig'][pve_idx, :],
                             mean_results[lfi_methods[method_idx]]['nsig'][pve_idx, :] + \
                                 sd_results[lfi_methods[method_idx]]['nsig'][pve_idx, :],
                                 color = 'red', alpha=0.3)
        axs[method_idx].set_xlabel(r'Correlation ($\rho$)', fontsize=24)
        # x-axis ticks should be rho values
        axs[method_idx].set_xticks(range(len(rhos)), rhos)
        # make xaxis label big
        axs[method_idx].tick_params(axis='both', labelsize=20)
        
        # set y-axis ticks at 30, 40, 50, 60, 70
        if pves[pve_idx] == 0.1:
            axs[method_idx].set_yticks([30, 40, 50, 60, 70])
        else:
            axs[method_idx].set_yticks([10, 20, 30, 40, 50, 60, 70])
        # y-axis label should be Average Ranking
        if method_idx == 0:
            axs[method_idx].set_ylabel("Average Rank", fontsize=24)
        axs[method_idx].set_title(titles[lfi_methods[method_idx]], fontsize=30, fontweight='bold')
    title_font_properties = font_manager.FontProperties(weight='bold', size=25)
    plt.legend(title = "Feature Group", loc = "upper left", bbox_to_anchor=(1,0.7),
               fontsize=22, title_fontproperties = title_font_properties, frameon=False)
    # fig.text(0.06, 0.5, "RF Model", ha='center', va='center', rotation=90, fontsize=24, fontstyle='italic', fontweight='bold')
    gb_plots.append(plt.gcf())
    # plt.savefig("images/bigger/correlation.png", format='png', bbox_inches='tight')
    # plt.savefig("images/bigger/correlation.pdf", format='pdf', bbox_inches='tight')
    plt.show()